# Lọc FIR thời gian thực cho tín hiệu microphone (So sánh tín hiệu vào và ra)

Chương trình này **thu âm tín hiệu microphone theo thời gian thực**, áp dụng **bộ lọc FIR dạng streaming**, và hiển thị **tín hiệu gốc (input)** và **tín hiệu sau lọc (output)** trên **hai cửa sổ đồ thị riêng biệt**.

Mục tiêu thiết kế là **đúng về mặt xử lý tín hiệu** và **độ trễ ổn định**, không chạy theo song song hoá không cần thiết.

---

## 1. Tổng quan hệ thống

Luồng xử lý tín hiệu:

```

Microphone → Bộ đệm âm thanh (CHUNK mẫu)
→ Bộ lọc FIR (có trạng thái)
→ Hiển thị miền thời gian

````

Hai cửa sổ hiển thị:
- **Cửa sổ 1**: Tín hiệu microphone gốc
- **Cửa sổ 2**: Tín hiệu sau lọc FIR

---

## 2. Cấu hình âm thanh

```python
FORMAT   = pyaudio.paInt16
CHANNELS = 1
RATE     = 44100
CHUNK    = 1024
````

* **FORMAT**: PCM 16-bit có dấu (chuẩn của soundcard)
* **CHANNELS**: Mono
* **RATE**: Tần số lấy mẫu (Hz)
* **CHUNK**: Số mẫu trong mỗi khối xử lý

### Lưu ý về độ trễ

Độ trễ tối thiểu xấp xỉ:

```
Độ trễ ≈ CHUNK / RATE
```

Ví dụ:

* `1024 / 44100 ≈ 23 ms`
* `512 / 44100 ≈ 11.6 ms`

Giảm `CHUNK` → giảm độ trễ nhưng tăng tải CPU.

---

## 3. Thiết kế bộ lọc FIR

```python
numtaps = 257
cutoff_hz = 3000

b = signal.firwin(
    numtaps,
    cutoff_hz,
    fs=RATE,
    window="hamming"
)
```

* **Loại lọc**: FIR thông thấp (low-pass)
* **Cửa sổ**: Hamming (giảm side-lobe tốt)
* **Tần số cắt**: 3 kHz
* **numtaps**: Quyết định độ dốc biên và độ suy giảm dải chắn

---

## 4. Trạng thái bộ lọc FIR (rất quan trọng)

```python
zi = np.zeros(len(b) - 1, dtype=np.float32)
```

Bộ lọc FIR **có bộ nhớ**.
Nếu không lưu trạng thái giữa các khối:

* Tín hiệu sẽ bị đứt đoạn
* Xuất hiện tiếng “click”
* Kết quả lọc sai

`zi` lưu các mẫu trễ để nối liên tục giữa các `CHUNK`.

---

## 5. Mở luồng âm thanh

```python
stream = p.open(
    format=FORMAT,
    channels=CHANNELS,
    rate=RATE,
    input=True,
    frames_per_buffer=CHUNK
)
```

* Mở luồng thu âm
* Dữ liệu trả về là **byte thô**
* Mỗi lần đọc được đúng `CHUNK` mẫu

---

## 6. Thiết lập hiển thị

Tạo **hai cửa sổ Matplotlib độc lập**.

### Cửa sổ tín hiệu vào

```python
fig_in, ax_in = plt.subplots()
line_in, = ax_in.plot(x, np.zeros(CHUNK))
ax_in.set_title("Microphone Input")
```

### Cửa sổ tín hiệu ra

```python
fig_out, ax_out = plt.subplots()
line_out, = ax_out.plot(x, np.zeros(CHUNK))
ax_out.set_title("FIR Filtered Output")
```

* Cùng trục thời gian
* Tách cửa sổ để dễ so sánh

---

## 7. Vòng lặp xử lý thời gian thực

```python
data = stream.read(CHUNK, exception_on_overflow=False)
```

* Đọc dữ liệu từ bộ đệm âm thanh
* Tránh lỗi tràn buffer khi CPU bận

---

## 8. Chuyển đổi dữ liệu

```python
x_int16 = np.frombuffer(data, dtype=np.int16)
x_f = x_int16.astype(np.float32)
```

* Dữ liệu gốc ở dạng `int16`
* FIR xử lý tốt hơn ở dạng `float`

---

## 9. Lọc FIR dạng streaming

```python
y_f, zi = signal.lfilter(b, [1.0], x_f, zi=zi)
```

* Lọc khối hiện tại
* Cập nhật trạng thái `zi` cho khối kế tiếp
* Đảm bảo tín hiệu **liên tục theo thời gian**

---

## 10. Chuyển về dạng hiển thị

```python
y_int16 = np.clip(y_f, -32768, 32767).astype(np.int16)
```

* Giới hạn biên độ tránh tràn số
* Đưa về định dạng 16-bit

---

## 11. Cập nhật đồ thị

```python
line_in.set_ydata(x_int16)
line_out.set_ydata(y_int16)

fig_in.canvas.draw()
fig_in.canvas.flush_events()

fig_out.canvas.draw()
fig_out.canvas.flush_events()
```

* Cập nhật cả hai cửa sổ
* Sử dụng chế độ interactive (`plt.ion()`)

> Lưu ý: vẽ đồ thị là phần **chậm nhất** trong toàn bộ hệ thống.

---

## 12. Vì sao không dùng song song hoá

* FIR cho mỗi `CHUNK` rất nhẹ
* Đa tiến trình gây thêm độ trễ do copy dữ liệu
* Matplotlib + I/O mới là nút cổ chai
* Thời gian thực cần **độ trễ ổn định**, không phải throughput cao

**Thứ tự tối ưu đúng:**

1. Giảm `CHUNK`
2. Giảm tần suất vẽ đồ thị
3. Tách phần hiển thị sang luồng khác (nâng cao)

---

## 13. Có thể mở rộng thêm

* FIR thông cao / thông dải
* Xuất âm thanh ra loa
* Hiển thị miền tần số (FFT)
* PyAudio callback (độ trễ thấp hơn)
* FIR dài bằng FFT overlap-save

---

## Kết luận

Chương trình này minh hoạ đúng:

* Lọc FIR thời gian thực
* Quản lý trạng thái bộ lọc
* So sánh trực quan input và output
* Độ trễ ổn định, dễ kiểm soát


```
```


In [7]:
import numpy as np
import pyaudio
import matplotlib.pyplot as plt
from scipy import signal

%matplotlib tk

# ======================
# Audio parameters
# ======================
FORMAT   = pyaudio.paInt16
CHANNELS = 1
RATE     = 44100
CHUNK    = 1024

# 1 second buffer
BUFFER_SEC = 0.1
BUFFER_LEN = int(RATE * BUFFER_SEC)

# ======================
# FIR filter
# ======================
NUMTAPS   = 257
CUTOFF_HZ = 3000

b = signal.firwin(NUMTAPS, CUTOFF_HZ, fs=RATE)
a = [1.0]


# ======================
# PyAudio
# ======================
p = pyaudio.PyAudio()
stream = p.open(
    format=FORMAT,
    channels=CHANNELS,
    rate=RATE,
    input=True,
    frames_per_buffer=CHUNK
)

# ======================
# Rolling buffers (1 s)
# ======================
buf_in  = np.zeros(BUFFER_LEN, dtype=np.int16)
buf_out = np.zeros(BUFFER_LEN, dtype=np.int16)

t_axis = np.linspace(-BUFFER_SEC, 0, BUFFER_LEN)

# ======================
# Plot setup
# ======================
plt.ion()

fig_in, ax_in = plt.subplots()
line_in, = ax_in.plot(t_axis, buf_in)
ax_in.set_title("Microphone Input (last 1 s)")
ax_in.set_xlabel("Time (seconds)")
ax_in.set_xlim(-BUFFER_SEC, 0)

fig_out, ax_out = plt.subplots()
line_out, = ax_out.plot(t_axis, buf_out)
ax_out.set_title("FIR Filtered Output (last 1 s)")
ax_out.set_xlabel("Time (seconds)")
ax_out.set_xlim(-BUFFER_SEC, 0)

# ======================
# RMS auto zoom
# ======================
def rms(x):
    return np.sqrt(np.mean(x.astype(np.float32)**2) + 1e-12)

ylim_smooth = 10000
SMOOTH = 0.9
MULT   = 4.0
MIN_Y  = 300

# ======================
# Main loop
# ======================
try:
    while True:
        data = stream.read(CHUNK, exception_on_overflow=False)

        x_int16 = np.frombuffer(data, dtype=np.int16)
        x_f = x_int16.astype(np.float32)

        # Filtering
        zi = signal.lfilter_zi(b, a) * x_f[0]
        y_f, zi = signal.lfilter(b, a, x_f, zi=zi)
        y_int16 = np.clip(y_f, -32768, 32767).astype(np.int16)

        # ---- roll buffers ----
        buf_in[:-CHUNK]  = buf_in[CHUNK:]
        buf_in[-CHUNK:]  = x_int16

        buf_out[:-CHUNK] = buf_out[CHUNK:]
        buf_out[-CHUNK:] = y_int16

        # ---- auto scale ----
        r = max(rms(buf_in), rms(buf_out))
        target = max(MIN_Y, MULT * r)
        ylim_smooth = SMOOTH * ylim_smooth + (1 - SMOOTH) * target

        ax_in.set_ylim(-ylim_smooth, ylim_smooth)
        ax_out.set_ylim(-ylim_smooth, ylim_smooth)

        # ---- update plots ----
        line_in.set_ydata(buf_in)
        line_out.set_ydata(buf_out)

        fig_in.canvas.draw()
        fig_in.canvas.flush_events()

        fig_out.canvas.draw()
        fig_out.canvas.flush_events()

except KeyboardInterrupt:
    print("Stopped")

finally:
    stream.stop_stream()
    stream.close()
    p.terminate()
    plt.ioff()
    plt.show()


Stopped


In [ ]:
import numpy as np
import pyaudio
import matplotlib.pyplot as plt
from scipy import signal

%matplotlib tk

# ======================
# Audio parameters
# ======================
FORMAT = pyaudio.paInt16
CHANNELS = 1
RATE = 44100
CHUNK = 1024

BUFFER_SEC = 0.1
BUFFER_LEN = int(RATE * BUFFER_SEC)

# ======================
# IIR Butterworth Filter
# ======================
ORDER = 4
CUTOFF_HZ = 3000

# Thiết kế dạng SOS (ổn định)
sos = signal.butter(ORDER, CUTOFF_HZ, btype="lowpass", fs=RATE, output="sos")

# Khởi tạo trạng thái filter
zi = signal.sosfilt_zi(sos)

# ======================
# PyAudio
# ======================
p = pyaudio.PyAudio()
stream = p.open(
    format=FORMAT, channels=CHANNELS, rate=RATE, input=True, frames_per_buffer=CHUNK
)

# ======================
# Rolling buffers
# ======================
buf_in = np.zeros(BUFFER_LEN, dtype=np.int16)
buf_out = np.zeros(BUFFER_LEN, dtype=np.int16)

t_axis = np.linspace(-BUFFER_SEC, 0, BUFFER_LEN)

# ======================
# Plot
# ======================
plt.ion()

# fig, (ax1, ax2) = plt.subplots(2, 1)

# (line1,) = ax1.plot(t_axis, buf_in)
# ax1.set_title("Mic Input")
# ax1.set_xlim(-BUFFER_SEC, 0)

# (line2,) = ax2.plot(t_axis, buf_out)
# ax2.set_title("IIR Butterworth Output")
# ax2.set_xlim(-BUFFER_SEC, 0)

# ======================
# Plot (2 canvas riêng)
# ======================
plt.ion()

# ---- Figure 1: Input ----
fig_in, ax_in = plt.subplots()
(line_in,) = ax_in.plot(t_axis, buf_in)
ax_in.set_title("Microphone Input (Realtime)")
ax_in.set_xlabel("Time (seconds)")
ax_in.set_xlim(-BUFFER_SEC, 0)

# ---- Figure 2: Output ----
fig_out, ax_out = plt.subplots()
(line_out,) = ax_out.plot(t_axis, buf_out)
ax_out.set_title("IIR Butterworth Output (Realtime)")
ax_out.set_xlabel("Time (seconds)")
ax_out.set_xlim(-BUFFER_SEC, 0)


# ======================
# RMS scaling
# ======================
def rms(x):
    return np.sqrt(np.mean(x.astype(np.float32) ** 2) + 1e-12)


ylim_smooth = 10000
SMOOTH = 0.9
MULT = 4.0
MIN_Y = 300

# ======================
# Main Loop
# ======================
try:
    while True:
        data = stream.read(CHUNK, exception_on_overflow=False)
        x_int16 = np.frombuffer(data, dtype=np.int16)
        x_float = x_int16.astype(np.float32)

        # ==== IIR Filtering (GIỮ TRẠNG THÁI) ====
        y_float, zi = signal.sosfilt(sos, x_float, zi=zi)

        y_int16 = np.clip(y_float, -32768, 32767).astype(np.int16)

        # ---- Roll buffers ----
        buf_in[:-CHUNK] = buf_in[CHUNK:]
        buf_in[-CHUNK:] = x_int16

        buf_out[:-CHUNK] = buf_out[CHUNK:]
        buf_out[-CHUNK:] = y_int16

        # ---- Auto scale ----
        r = max(rms(buf_in), rms(buf_out))
        target = max(MIN_Y, MULT * r)
        ylim_smooth = SMOOTH * ylim_smooth + (1 - SMOOTH) * target

        ax_in.set_ylim(-ylim_smooth, ylim_smooth)
        ax_out.set_ylim(-ylim_smooth, ylim_smooth)

        # ---- Update plots ----
        line_in.set_ydata(buf_in)
        line_out.set_ydata(buf_out)

        fig_in.canvas.draw()
        fig_in.canvas.flush_events()

        fig_out.canvas.draw()
        fig_out.canvas.flush_events()

except KeyboardInterrupt:
    print("Stopped")

finally:
    stream.stop_stream()
    stream.close()
    p.terminate()
    plt.ioff()
    plt.show()